# Gender prediction — submission notebook
Single-pass: features → OOF gender priors → train one LGBM on full train → predict test → `submission.csv`.
No Optuna, no CV. Uses params already found via prior tuning.

In [1]:
# Colab setup (uncomment if needed)
# !pip install -q lightgbm
# from google.colab import drive; drive.mount('/content/drive')
# !mkdir -p data && cp /content/drive/MyDrive/hackathon/*.csv data/

In [2]:
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold

DATA_DIR = 'data'
SEED = 42
N_BOOST = 850
PARAMS = {
    'objective': 'binary', 'metric': 'auc', 'verbose': -1,
    'learning_rate': 0.018, 'num_leaves': 44, 'min_child_samples': 36,
    'feature_fraction': 0.737, 'bagging_fraction': 0.836, 'bagging_freq': 1,
    'lambda_l1': 0.20, 'lambda_l2': 2.57, 'seed': SEED,
}
T0 = time.time()

In [3]:
g = pd.read_csv(f'{DATA_DIR}/gender.csv')
trx = pd.read_csv(f'{DATA_DIR}/trx.csv')
tr = g[g.gender.notna()].copy(); tr['gender'] = tr['gender'].astype(int)
te = g[g.gender.isna()].copy()
print(f'train: {len(tr)}  test: {len(te)}')

train: 6400  test: 2000


In [4]:
def _mode(x):
    v = x.value_counts(); return v.index[0] if len(v) else -1

trx = trx.sort_values(['cid', 'dt']).reset_index(drop=True)
pos = trx[trx.amt > 0]; neg = trx[trx.amt < 0]
g_ = trx.groupby('cid')
feat = pd.DataFrame(index=g_.size().index)
feat['n_trx'] = g_.size()
feat['n_days'] = g_.dt.nunique()
feat['dt_min'] = g_.dt.min(); feat['dt_max'] = g_.dt.max()
feat['active_span'] = feat['dt_max'] - feat['dt_min'] + 1
feat['trx_per_day'] = feat['n_trx'] / feat['active_span']
daily = trx.groupby(['cid','dt']).size().rename('c').reset_index()
dpd = daily.groupby('cid').c
feat['dpd_mean'] = dpd.mean(); feat['dpd_std'] = dpd.std()

a = g_.amt
feat['amt_mean']=a.mean(); feat['amt_std']=a.std(); feat['amt_med']=a.median()
feat['amt_min']=a.min(); feat['amt_max']=a.max(); feat['amt_sum']=a.sum()
feat['amt_skew']=a.skew(); feat['amt_kurt']=a.apply(lambda x: x.kurt() if len(x)>3 else 0.0)
feat['amt_q05']=a.quantile(0.05); feat['amt_q25']=a.quantile(0.25)
feat['amt_q75']=a.quantile(0.75); feat['amt_q95']=a.quantile(0.95)

abs_a = trx.amt.abs()
feat['amt_abs_mean']=abs_a.groupby(trx.cid).mean()
feat['amt_abs_sum']=abs_a.groupby(trx.cid).sum()
feat['amt_abs_max']=abs_a.groupby(trx.cid).max()
feat['concentration']=feat['amt_abs_max']/(feat['amt_abs_sum']+1)
log_a = np.log1p(abs_a)
feat['log_amt_mean']=log_a.groupby(trx.cid).mean()
feat['log_amt_std']=log_a.groupby(trx.cid).std()

pg=pos.groupby('cid').amt; ng=neg.groupby('cid').amt
feat['pos_n']=pg.size().reindex(feat.index).fillna(0)
feat['pos_mean']=pg.mean().reindex(feat.index)
feat['pos_sum']=pg.sum().reindex(feat.index).fillna(0)
feat['neg_n']=ng.size().reindex(feat.index).fillna(0)
feat['neg_mean']=ng.mean().reindex(feat.index)
feat['neg_sum']=ng.sum().reindex(feat.index).fillna(0)
feat['cd_ratio']=feat['pos_n']/(feat['neg_n']+1)
feat['cd_amt_ratio']=feat['pos_sum']/(feat['neg_sum'].abs()+1)

feat['n_mcc']=g_.mcc.nunique(); feat['n_ttc']=g_.ttc.nunique(); feat['n_tid']=g_.tid.nunique()
feat['mcc_mode']=g_.mcc.agg(_mode); feat['ttc_mode']=g_.ttc.agg(_mode)
mcl = trx.groupby(['cid','mcc']).size().rename('c').reset_index().sort_values(['cid','c'], ascending=[True,False])
feat['mcc_top3_share']=mcl.groupby('cid').head(3).groupby('cid').c.sum()/feat['n_trx']

def share(mask, name):
    s = trx[mask].groupby('cid').size().reindex(feat.index).fillna(0)
    feat[name] = s / feat['n_trx']
share(trx.mcc.between(56,61), 'sh_food')
share(trx.mcc.between(5,18), 'sh_transport')
share(trx.mcc.between(69,79), 'sh_clothing')
share(trx.mcc.between(153,165), 'sh_entert')
share(trx.ttc.between(10,19), 'sh_cash')
share(trx.ttc.between(25,54), 'sh_c2c')
share(trx.ttc==23, 'sh_online')
share(trx.ttc.isin([1,4,6]), 'sh_pos')

diffs = trx.groupby('cid').dt.diff(); diffs.index = trx.cid.values
feat['gap_mean']=diffs.groupby(level=0).mean(); feat['gap_std']=diffs.groupby(level=0).std()
trx['dow']=trx.dt%7; trx['is_we']=trx.dow.isin([5,6]).astype(int)
feat['sh_weekend']=trx.groupby('cid').is_we.mean()
h = pd.to_datetime(trx.tm, format='%H:%M:%S', errors='coerce').dt.hour
feat['hour_mean']=h.groupby(trx.cid).mean()
feat['hour_std']=h.groupby(trx.cid).std()
feat['sh_night']=((h<6)|(h>=22)).groupby(trx.cid).mean()
print(f'base done: {feat.shape[1]} cols')

base done: 54 cols


In [5]:
trx['amt_abs'] = trx.amt.abs()
mcc_cnt = trx.groupby(['cid','mcc']).size().unstack(fill_value=0)
mcc_sum = trx.groupby(['cid','mcc']).amt_abs.sum().unstack(fill_value=0)
mcc_share = mcc_sum.div(mcc_sum.sum(axis=1).replace(0,1), axis=0)
probs = mcc_cnt.div(mcc_cnt.sum(axis=1).replace(0,1), axis=0)
mcc_entropy = -(probs * np.log(probs+1e-12)).sum(axis=1).rename('mcc_entropy')
mcc_piv = pd.concat([mcc_cnt.add_prefix('mcc_cnt_'), mcc_sum.add_prefix('mcc_sum_'),
                     mcc_share.add_prefix('mcc_share_'), mcc_entropy], axis=1)

top_ttc = trx.ttc.value_counts().head(30).index
tsub = trx[trx.ttc.isin(top_ttc)]
ttc_cnt = tsub.groupby(['cid','ttc']).size().unstack(fill_value=0).reindex(sorted(trx.cid.unique()), fill_value=0)
ttc_share = ttc_cnt.div(ttc_cnt.sum(axis=1).replace(0,1), axis=0)
ttc_piv = pd.concat([ttc_cnt.add_prefix('ttc_cnt_'), ttc_share.add_prefix('ttc_share_')], axis=1)

dow_hist = pd.crosstab(trx.cid, trx.dow, normalize='index')
dow_hist.columns = [f'dow_sh_{c}' for c in dow_hist.columns]

mid = (trx.dt.min() + trx.dt.max()) / 2
early = trx[trx.dt <= mid]; late = trx[trx.dt > mid]
cids_all = sorted(trx.cid.unique())
def _agg(df, suf):
    a = df.groupby('cid').agg(n=('amt','size'), s=('amt','sum'), nm=('mcc','nunique'))
    a.columns = [f'{c}_{suf}' for c in ['n','s','nm']]
    return a.reindex(cids_all, fill_value=0)
tmp = pd.concat([_agg(early,'early'), _agg(late,'late')], axis=1)
tmp['n_late_early_r']=tmp['n_late']/(tmp['n_early']+1)
tmp['s_late_early_r']=tmp['s_late']/(tmp['s_early'].abs()+1)
tmp['nm_late_early_r']=tmp['nm_late']/(tmp['nm_early']+1)

feat = pd.concat([feat, mcc_piv, ttc_piv, dow_hist, tmp], axis=1).reset_index()
if feat.columns[0] != 'cid':
    feat = feat.rename(columns={feat.columns[0]: 'cid'})
print(f'feat shape: {feat.shape}')

feat shape: (8400, 684)


In [6]:
tr_cids = tr.cid.values; te_cids = te.cid.values
y = tr.gender.values.astype(int)
cnt_tr = mcc_cnt.loc[tr_cids].values.astype(np.float32)
cnt_te = mcc_cnt.loc[te_cids].values.astype(np.float32)
used_tr = (cnt_tr > 0).astype(np.float32)
yf = y.astype(np.float32); n_tr = len(tr_cids)

oof_w=np.zeros(n_tr,dtype=np.float32); oof_f=np.zeros(n_tr,dtype=np.float32); oof_m=np.zeros(n_tr,dtype=np.float32)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for tr_i, va_i in skf.split(np.zeros(n_tr), y):
    u = used_tr[tr_i]; ucnt = u.sum(axis=0)
    rates = np.where(ucnt>=5, (u.T @ yf[tr_i])/np.maximum(ucnt,1), 0.5)
    c = cnt_tr[va_i]; tot = c.sum(axis=1).clip(min=1)
    oof_w[va_i] = (c*rates).sum(axis=1)/tot
    oof_f[va_i] = c[:, rates<0.4].sum(axis=1)/tot
    oof_m[va_i] = c[:, rates>0.6].sum(axis=1)/tot
ucnt = used_tr.sum(axis=0)
rates = np.where(ucnt>=5, (used_tr.T @ yf)/np.maximum(ucnt,1), 0.5)
tot = cnt_te.sum(axis=1).clip(min=1)
te_w=(cnt_te*rates).sum(axis=1)/tot
te_f=cnt_te[:, rates<0.4].sum(axis=1)/tot
te_m=cnt_te[:, rates>0.6].sum(axis=1)/tot

gp = pd.concat([
    pd.DataFrame({'cid':tr_cids,'gp_w':oof_w,'gp_fem':oof_f,'gp_mal':oof_m}),
    pd.DataFrame({'cid':te_cids,'gp_w':te_w, 'gp_fem':te_f, 'gp_mal':te_m}),
], ignore_index=True)
feat = feat.merge(gp, on='cid', how='left')
print('priors merged')

priors merged


In [7]:
tr_f = tr.merge(feat, on='cid', how='left')
te_f = te.merge(feat, on='cid', how='left')
TOP5 = ['gp_w','gp_fem','gp_mal','sh_clothing','mcc_share_116']
def add_ix(df, t):
    eps=1e-6; a,b,c,d,e = t[:5]
    df[f'ix_{a}_div_{b}']=df[a]/(df[b]+eps); df[f'ix_{a}_div_{c}']=df[a]/(df[c]+eps)
    df[f'ix_{b}_div_{c}']=df[b]/(df[c]+eps); df[f'ix_{a}_div_{d}']=df[a]/(df[d]+eps)
    df[f'ix_{a}_div_{e}']=df[a]/(df[e]+eps)
    df[f'ix_{a}_x_{b}']=df[a]*df[b]; df[f'ix_{a}_x_{c}']=df[a]*df[c]
    df[f'ix_{b}_x_{c}']=df[b]*df[c]; df[f'ix_{a}_x_{d}']=df[a]*df[d]
    df[f'ix_{a}_x_{e}']=df[a]*df[e]
    return df
tr_f = add_ix(tr_f, TOP5); te_f = add_ix(te_f, TOP5)
cols = [c for c in tr_f.columns if c not in ['cid','gender']]
X_tr = tr_f[cols]; X_te = te_f[cols]
print(f'X_tr: {X_tr.shape}  X_te: {X_te.shape}')

X_tr: (6400, 696)  X_te: (2000, 696)


In [8]:
t1 = time.time()
dtr = lgb.Dataset(X_tr, y)
m = lgb.train(PARAMS, dtr, num_boost_round=N_BOOST, callbacks=[lgb.log_evaluation(0)])
pred = m.predict(X_te)
print(f'fit+predict: {time.time()-t1:.1f}s')

sub = pd.DataFrame({'cid': te_f.cid.values, 'gender': pred})
sub.to_csv('submission.csv', index=False)
print(f'submission.csv  rows={len(sub)}')
print(sub.head())
print(f'\nTOTAL: {time.time()-T0:.1f}s')

fit+predict: 10.5s
submission.csv  rows=2000
   cid    gender
0    0  0.330964
1    1  0.271218
2    2  0.960006
3    3  0.813393
4    4  0.193113

TOTAL: 20.7s
